# blockchainkit quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cpoli/blockchainkit/blob/main/notebooks/quickstart.ipynb)

blockchainkit follows each idea in a blockchain from its history and mathematics to a small Python API you can poke at. This notebook takes one payment from a signature to a mined block in about ten minutes: sign, commit, mine, choose between forks, and execute a program. Every key and nonce here is a fixed teaching value; never reuse one for anything real.

In [ ]:
%pip install -q blockchainkit

In [ ]:
import matplotlib.pyplot as plt

import blockchainkit as bk

print(bk.__version__)

## 1. Sign a payment

A private key is a number; the public key is that number times a curve's generator point. A Schnorr signature proves knowledge of the private key for *these exact bytes*: change one byte and verification fails.

In [ ]:
alice_key, bob_key = 7, 11
alice = bk.structures.address(bk.crypto.public_key(alice_key))
bob = bk.structures.address(bk.crypto.public_key(bob_key))

payment = bk.structures.Transaction(bk.crypto.public_key(alice_key), bob, 25, 0)
payment = payment.signed(alice_key, signing_nonce=17)  # Never reuse a signing nonce.
print("valid:", payment.is_valid())
print("tampered:", bk.structures.Transaction(payment.sender, bob, 26, 0, signature=payment.signature).is_valid())

## 2. Commit to many payments with one hash

A Merkle root commits to an ordered list. Proving one item takes only the orange siblings: a logarithmic number of hashes.

In [ ]:
from blockchainkit.structures.visualizers import plot_merkle_tree

leaves = [f"payment {i}".encode() for i in range(6)]
tree = bk.structures.MerkleTree(leaves)
print("proof verifies:", bk.structures.verify_proof(leaves[3], tree.proof(3), tree.root))
plot_merkle_tree(tree, highlight=3);

## 3. Mine a block

Proof of work searches nonces until the block hash falls below a target. Each extra required zero bit doubles the expected number of attempts.

In [ ]:
from blockchainkit.consensus.visualizers import plot_mining_trials

genesis = bk.consensus.mine(bk.structures.Block(difficulty=6)).block
chain = bk.structures.Blockchain(genesis, bk.structures.Ledger({alice: 100}))
block = bk.structures.Block(genesis.hash, (payment,), height=1, timestamp=1, difficulty=6)
chain.add(bk.consensus.mine(block).block)
print(chain)
print("Bob's balance:", chain.state.balances[bob])

difficulties = [d for d in range(2, 11) for _ in range(5)]
attempts = [
    bk.consensus.mine(bk.structures.Block(timestamp=i, difficulty=d)).attempts
    for i, d in enumerate(difficulties)
]
plot_mining_trials(difficulties, attempts);

## 4. Choose between two histories

Two valid chains can exist at once. Peers follow the one with the most cumulative work, so a longer fork *reorganizes* the chain and the ledger state with it.

In [ ]:
from blockchainkit.structures.visualizers import plot_block_tree

rival = bk.consensus.mine(bk.structures.Block(genesis.hash, height=1, timestamp=2, difficulty=6)).block
rival_child = bk.consensus.mine(bk.structures.Block(rival.hash, height=2, timestamp=3, difficulty=6)).block
chain.add(rival)
chain.add(rival_child)
print("Bob's balance after the reorganization:", chain.state.balances.get(bob, 0))
plot_block_tree(chain);

## 5. Execute a program with a gas budget

Every node must reach the same state, so execution is deterministic and bounded: a gas limit stops even an infinite loop.

In [ ]:
from blockchainkit.vm.visualizers import plot_execution_trace

program = [("LOAD", 0), ("PUSH", 5), ("ADD", None), ("STORE", 0), ("STOP", None)]
result = bk.vm.execute(program, storage={0: 10}, trace=True)
plot_execution_trace(result.trace);

try:
    bk.vm.execute([("JMP", 0)], gas_limit=100)
except bk.vm.VMError as error:
    print("stopped:", error)

## Where next

- The [history pages](https://cpoli.github.io/blockchainkit/history/index.html) trace each idea to its original paper.
- The [example gallery](https://cpoli.github.io/blockchainkit/examples/index.html) has one experiment per idea, each with an exercise.
- The [guided course](https://cpoli.github.io/blockchainkit/tutorials/course.html) gives a learning order.